# Urdu Prompting Study — Live Demo
**Course:** CS4063 NLP | **Task:** Roman-Urdu Sentiment Classification

This notebook walks through real model inputs and outputs from the experiment.
No GPU needed — all results are pre-computed.

In [ ]:
import json, pathlib, pandas as pd
from IPython.display import display, HTML

ROOT = pathlib.Path('.')   # run from project root
PRED = ROOT / 'results' / 'predictions'

EN_SYS  = ('You are a precise sentiment classifier for Urdu text. '
            'Respond with exactly one word from: negative, neutral, positive. '
            'Do not add punctuation or explanation.')
EN_INS  = 'Classify the sentiment of the following Urdu text. Answer with one word: negative, neutral, or positive.'
EN_COT  = ('Classify the sentiment of the following Urdu text. '
            'First briefly reason step by step in English, then on a new line '
            "write 'Answer: <label>' where <label> is one of: negative, neutral, positive.")
UR_SYS  = ('\u0622\u067e \u0627\u06cc\u06a9 \u062f\u0631\u0633\u062a \u0627\u0631\u062f\u0648 \u062c\u0630\u0628\u0627\u062a \u06a9\u06cc \u062f\u0631\u062c\u06c1 \u0628\u0646\u062f\u06cc \u06a9\u0631\u0646\u06d2 \u0648\u0627\u0644\u06d2 \u06c1\u06cc\u06ba\u06d4 '
            '\u0635\u0631\u0641 \u0627\u06cc\u06a9 \u0644\u0641\u0638 \u0633\u06d2 \u062c\u0648\u0627\u0628 \u062f\u06cc\u06ba: \u0645\u0646\u0641\u06cc\u060c \u063a\u06cc\u0631 \u062c\u0627\u0646\u0628\u062f\u0627\u0631\u060c \u06cc\u0627 \u0645\u062b\u0628\u062a\u06d4 '
            '\u06a9\u0648\u0626\u06cc \u0648\u0636\u0627\u062d\u062a \u06cc\u0627 \u0639\u0644\u0627\u0645\u0627\u062a \u0634\u0627\u0645\u0644 \u0646\u06c1 \u06a9\u0631\u06cc\u06ba\u06d4')
UR_INS  = ('\u062f\u0631\u062c \u0630\u06cc\u0644 \u0627\u0631\u062f\u0648 \u0645\u062a\u0646 \u06a9\u06d2 \u062c\u0630\u0628\u0627\u062a \u06a9\u06cc \u062f\u0631\u062c\u06c1 \u0628\u0646\u062f\u06cc \u06a9\u0631\u06cc\u06ba\u06d4 '
            '\u0635\u0631\u0641 \u0627\u06cc\u06a9 \u0644\u0641\u0638 \u0633\u06d2 \u062c\u0648\u0627\u0628 \u062f\u06cc\u06ba: \u0645\u0646\u0641\u06cc\u060c \u063a\u06cc\u0631 \u062c\u0627\u0646\u0628\u062f\u0627\u0631\u060c \u06cc\u0627 \u0645\u062b\u0628\u062a\u06d4')
UR_COT  = ('\u062f\u0631\u062c \u0630\u06cc\u0644 \u0627\u0631\u062f\u0648 \u0645\u062a\u0646 \u06a9\u06d2 \u062c\u0630\u0628\u0627\u062a \u06a9\u06cc \u062f\u0631\u062c\u06c1 \u0628\u0646\u062f\u06cc \u06a9\u0631\u06cc\u06ba\u06d4 '
            '\u067e\u06c1\u0644\u06d2 \u0645\u062e\u062a\u0635\u0631 \u0642\u062f\u0645 \u0628\u06c1 \u0642\u062f\u0645 \u0627\u0631\u062f\u0648 \u0645\u06cc\u06ba \u0633\u0648\u0686\u06cc\u06ba\u060c \u067e\u06be\u0631 \u0646\u0626\u06cc \u0633\u0637\u0631 \u067e\u0631 \u0644\u06a9\u06be\u06cc\u06ba '
            "'\u062c\u0648\u0627\u0628: <\u0644\u06cc\u0628\u0644>' \u062c\u06c1\u0627\u06ba <\u0644\u06cc\u0628\u0644> \u0627\u0646 \u0645\u06cc\u06ba \u0633\u06d2 \u0627\u06cc\u06a9 \u06c1\u0648: \u0645\u0646\u0641\u06cc\u060c \u063a\u06cc\u0631 \u062c\u0627\u0646\u0628\u062f\u0627\u0631\u060c \u0645\u062b\u0628\u062a\u06d4")
UR_LBL  = {'negative':'\u0645\u0646\u0641\u06cc','neutral':'\u063a\u06cc\u0631 \u062c\u0627\u0646\u0628\u062f\u0627\u0631','positive':'\u0645\u062b\u0628\u062a'}

def build_full_prompt(variant, text, shots=None):
    shots = shots or []
    shot_block_en = '\n\n'.join(f"Text: {s['text']}\nAnswer: {s['label']}" for s in shots)
    shot_block_ur = '\n\n'.join(f"\u0645\u062a\u0646: {s['text']}\n\u062c\u0648\u0627\u0628: {UR_LBL[s['label']]}" for s in shots)
    if shot_block_en: shot_block_en += '\n\n'
    if shot_block_ur: shot_block_ur += '\n\n'
    templates = {
        'zs_en':  (EN_SYS, f"{EN_INS}\n\nText: {text}\nAnswer:"),
        'zs_ur':  (UR_SYS, f"{UR_INS}\n\n\u0645\u062a\u0646: {text}\n\u062c\u0648\u0627\u0628:"),
        'fs_en':  (EN_SYS, f"{EN_INS}\n\n{shot_block_en}Text: {text}\nAnswer:"),
        'fs_ur':  (UR_SYS, f"{UR_INS}\n\n{shot_block_ur}\u0645\u062a\u0646: {text}\n\u062c\u0648\u0627\u0628:"),
        'cot_en': (EN_SYS, f"{EN_COT}\n\nText: {text}"),
        'cot_ur': (UR_SYS, f"{UR_COT}\n\n\u0645\u062a\u0646: {text}"),
    }
    sys_msg, user_msg = templates[variant]
    return sys_msg, user_msg

SHOTS = [
    {'text': 'mujhe yeh pasand nahi', 'label': 'negative'},
    {'text': 'aj mausam theek hai',   'label': 'neutral'},
    {'text': 'bohat acha kaam kiya',  'label': 'positive'},
]

def load_preds(model_short, variant):
    path = PRED / f'{model_short}_{variant}.jsonl'
    return [json.loads(l) for l in path.read_text(encoding='utf-8').strip().splitlines()]

print('Setup complete')

---
## 1  Results Summary — all 12 cells at a glance

In [ ]:
df = pd.read_csv(ROOT / 'results' / 'metrics.csv')
df = df.rename(columns={'macro_f1':'Macro-F1','accuracy':'Accuracy',
                         'unknown_rate':'Unknown%','model':'Model','prompt':'Prompt'})
df['Unknown%'] = (df['Unknown%']*100).round(1)
df['Macro-F1'] = df['Macro-F1'].round(3)
df['Accuracy'] = df['Accuracy'].round(3)

def colour_f1(val):
    if isinstance(val, float):
        if val >= 0.5:  return 'background-color:#c6efce; color:#276221'
        if val >= 0.3:  return 'background-color:#ffeb9c; color:#9c5700'
        return 'background-color:#ffc7ce; color:#9c0006'
    return ''

styled = (df[['Model','Prompt','Accuracy','Macro-F1','Unknown%']]
            .style
            .map(colour_f1, subset=['Macro-F1'])
            .set_caption('Green >= 0.5  |  Yellow >= 0.3  |  Red < 0.3'))
display(styled)

---
## 2  Full Prompt → Raw Output → Prediction
### Change MODEL / VARIANT / IDX and re-run to explore any example

In [ ]:
# ── Change these three lines ──────────────────────────────
MODEL   = 'qwen15'   # qwen05 | qwen15
VARIANT = 'fs_en'    # zs_en | zs_ur | fs_en | fs_ur | cot_en | cot_ur
IDX     = 0          # 0 to 497
# ─────────────────────────────────────────────────────────

preds   = load_preds(MODEL, VARIANT)
example = preds[IDX]
shots   = SHOTS if VARIANT.startswith('fs_') else []
sys_msg, user_msg = build_full_prompt(VARIANT, example['text'], shots)

correct      = example['pred'] == example['gold']
badge        = 'CORRECT' if correct else 'WRONG'
badge_colour = '#c6efce' if correct else '#ffc7ce'

html = f"""
<div style='font-family:monospace;font-size:13px;line-height:1.6'>
  <h3>Model: <code>{MODEL}</code> | Variant: <code>{VARIANT}</code> | Example #{IDX}</h3>
  <div style='background:#e8eaf6;padding:10px;border-radius:6px;margin:6px 0'>
    <b>INPUT TEXT (Roman-Urdu)</b><br>
    <span style='font-size:15px'>{example['text']}</span>
  </div>
  <details open>
    <summary><b>SYSTEM PROMPT</b></summary>
    <pre style='background:#f5f5f5;padding:10px;border-radius:6px;white-space:pre-wrap'>{sys_msg}</pre>
  </details>
  <details open>
    <summary><b>USER PROMPT</b></summary>
    <pre style='background:#f5f5f5;padding:10px;border-radius:6px;white-space:pre-wrap'>{user_msg}</pre>
  </details>
  <div style='background:#fff9c4;padding:10px;border-radius:6px;margin:6px 0'>
    <b>MODEL RAW OUTPUT</b><br>
    <pre style='white-space:pre-wrap;margin:4px 0'>{example['raw']}</pre>
  </div>
  <table style='border-collapse:collapse;margin-top:8px;font-size:14px'>
    <tr>
      <td style='padding:6px 16px;background:#e3f2fd;border-radius:4px'><b>Gold:</b> {example['gold']}</td>
      <td style='padding:6px 16px;background:#e3f2fd;border-radius:4px'><b>Predicted:</b> {example['pred']}</td>
      <td style='padding:6px 16px;background:{badge_colour};border-radius:4px;font-weight:bold'>{badge}</td>
    </tr>
  </table>
</div>
"""
display(HTML(html))

---
## 3  English vs Urdu — Side-by-side for the SAME input

In [ ]:
# ── Change these ──────────────────
MODEL  = 'qwen05'  # qwen05 | qwen15
REGIME = 'zs'      # zs | fs | cot
IDX    = 0         # 0 to 497
# ──────────────────────────────────

en_var, ur_var = f'{REGIME}_en', f'{REGIME}_ur'
en_ex = load_preds(MODEL, en_var)[IDX]
ur_ex = load_preds(MODEL, ur_var)[IDX]

def bg(pred, gold):
    return '#c6efce' if pred == gold else '#ffc7ce'

html = f"""
<div style='font-family:monospace;font-size:13px'>
  <h3>Model: <code>{MODEL}</code> | Regime: <code>{REGIME}</code> | Example #{IDX}</h3>
  <div style='background:#e8eaf6;padding:10px;border-radius:6px;margin-bottom:10px'>
    <b>Input:</b> {en_ex['text']}<br><b>Gold label:</b> <b>{en_ex['gold']}</b>
  </div>
  <table style='width:100%;border-collapse:collapse'>
    <tr>
      <th style='width:50%;padding:8px;background:#1565c0;color:white;text-align:left'>English Prompt ({en_var})</th>
      <th style='width:50%;padding:8px;background:#4a148c;color:white;text-align:left'>Urdu Prompt ({ur_var})</th>
    </tr>
    <tr>
      <td style='padding:10px;border:1px solid #ddd;vertical-align:top'>
        <b>Raw output:</b><br>
        <pre style='background:#f5f5f5;padding:6px;white-space:pre-wrap;border-radius:4px'>{en_ex['raw'][:400]}</pre>
      </td>
      <td style='padding:10px;border:1px solid #ddd;vertical-align:top'>
        <b>Raw output:</b><br>
        <pre style='background:#f5f5f5;padding:6px;white-space:pre-wrap;border-radius:4px'>{ur_ex['raw'][:400]}</pre>
      </td>
    </tr>
    <tr>
      <td style='padding:8px;background:{bg(en_ex["pred"],en_ex["gold"])};text-align:center;font-size:15px;font-weight:bold'>
        Prediction: {en_ex['pred']} {'OK' if en_ex['pred']==en_ex['gold'] else 'WRONG'}
      </td>
      <td style='padding:8px;background:{bg(ur_ex["pred"],ur_ex["gold"])};text-align:center;font-size:15px;font-weight:bold'>
        Prediction: {ur_ex['pred']} {'OK' if ur_ex['pred']==ur_ex['gold'] else 'WRONG'}
      </td>
    </tr>
  </table>
</div>
"""
display(HTML(html))

---
## 4  Label Collapse — Urdu prompts predict the SAME label for every input

In [ ]:
from collections import Counter

print('=== URDU prompts ===')
for model in ['qwen05', 'qwen15']:
    for variant in ['zs_ur', 'fs_ur', 'cot_ur']:
        rows   = load_preds(model, variant)
        counts = Counter(r['pred'] for r in rows)
        total  = len(rows)
        bars   = ' | '.join(f"{k}: {v} ({v/total:.0%})" for k, v in counts.most_common())
        print(f"  {model:8} {variant:8}  ->  {bars}")
    print()

print('=== ENGLISH prompts (for comparison) ===')
for model in ['qwen05', 'qwen15']:
    for variant in ['zs_en', 'fs_en', 'cot_en']:
        rows   = load_preds(model, variant)
        counts = Counter(r['pred'] for r in rows)
        total  = len(rows)
        bars   = ' | '.join(f"{k}: {v} ({v/total:.0%})" for k, v in counts.most_common())
        print(f"  {model:8} {variant:8}  ->  {bars}")
    print()

---
## 5  Chain-of-Thought — English reasoning vs Urdu collapse

In [ ]:
# ── Change IDX to see a different example ──
IDX = 1   # try 0, 1, 2, 5
# ───────────────────────────────────────────

en_ex = load_preds('qwen15', 'cot_en')[IDX]
ur_ex = load_preds('qwen05', 'cot_ur')[IDX]

def bg(pred, gold):
    return '#c6efce' if pred == gold else '#ffc7ce'

html = f"""
<div style='font-family:monospace;font-size:13px'>
  <div style='background:#e8eaf6;padding:10px;border-radius:6px;margin-bottom:10px'>
    <b>Input:</b> {en_ex['text']}<br><b>Gold:</b> {en_ex['gold']}
  </div>
  <table style='width:100%;border-collapse:collapse'>
    <tr>
      <th style='padding:8px;background:#1565c0;color:white;text-align:left;width:50%'>
        Qwen 1.5B cot_en — English CoT reasoning
      </th>
      <th style='padding:8px;background:#b71c1c;color:white;text-align:left;width:50%'>
        Qwen 0.5B cot_ur — Urdu CoT (breaks down)
      </th>
    </tr>
    <tr>
      <td style='padding:10px;border:1px solid #ddd;vertical-align:top'>
        <pre style='white-space:pre-wrap;background:#f5f5f5;padding:8px;border-radius:4px'>{en_ex['raw']}</pre>
      </td>
      <td style='padding:10px;border:1px solid #ddd;vertical-align:top'>
        <pre style='white-space:pre-wrap;background:#f5f5f5;padding:8px;border-radius:4px'>{ur_ex['raw']}</pre>
      </td>
    </tr>
    <tr>
      <td style='padding:8px;background:{bg(en_ex["pred"],en_ex["gold"])};text-align:center;font-weight:bold'>
        Pred: {en_ex['pred']} {'OK' if en_ex['pred']==en_ex['gold'] else 'WRONG'}
      </td>
      <td style='padding:8px;background:{bg(ur_ex["pred"],ur_ex["gold"])};text-align:center;font-weight:bold'>
        Pred: {ur_ex['pred']} {'OK' if ur_ex['pred']==ur_ex['gold'] else 'WRONG'}
      </td>
    </tr>
  </table>
</div>
"""
display(HTML(html))

---
## 6  Best model (qwen15 fs_en) — correct and incorrect predictions

In [ ]:
rows  = load_preds('qwen15', 'fs_en')
wrong = [r for r in rows if r['pred'] != r['gold']]
right = [r for r in rows if r['pred'] == r['gold']]
print(f"qwen15 fs_en  |  Correct: {len(right)} ({len(right)/len(rows):.1%})  "
      f"|  Wrong: {len(wrong)} ({len(wrong)/len(rows):.1%})")

print("\n--- First 10 errors ---")
err_df = pd.DataFrame([
    {'Input': r['text'][:60], 'Gold': r['gold'], 'Predicted': r['pred']}
    for r in wrong[:10]
])
display(err_df.style.map(lambda v: 'color:red;font-weight:bold', subset=['Predicted']))

---
## 7  Statistical Significance — McNemar test results

In [ ]:
sig = pd.read_csv(ROOT / 'results' / 'significance.csv')

def sig_label(p):
    if p < 0.001: return '*** p<0.001'
    if p < 0.01:  return '**  p<0.01'
    if p < 0.05:  return '*   p<0.05'
    return 'n.s.'

sig['significance'] = sig['p_value'].apply(sig_label)
sig['winner'] = sig.apply(lambda r: 'English better' if r['b'] > r['c'] else 'Urdu better', axis=1)

def colour_sig(val):
    if 'n.s.' in str(val): return 'background:#ffc7ce'
    return 'background:#c6efce'

display(sig[['model','setting','b','c','p_value','significance','winner']]
        .style.map(colour_sig, subset=['significance']))